# R15-H212 - Clean-Conditions Recall Re-run (closing the H158 confound)

**Hypothesis**: re-running the H158 v2 arm (identity_stack v2, Bedrock Sonnet engine) under quiet
Bedrock - no concurrent wave-2 quota contention - yields failed-chunk warnings at or below the v1
arm's 76 and recall@16 >= 0.875, lifting H158 to CONFIRMED and green-lighting the v2 default flip.

**Registered measurement** (same H158 harness, `notebooks/h158_measure.py`):
- failed-chunk warnings = `extraction.warning` events in the ingest event log (H158: v1 76, v2 119)
- pure-seed evidence recall@16 over the 24-gold H34 probe set (H158: v1 0.875, v2 0.8333)
- SAME_AS precision proxy + false-merge count vs the 298-pair H101 benchmark (continuity)

**Registered bar**: warnings <= 76 AND recall@16 >= 0.875 -> H158 CONFIRMED + default flip approved.
DEGRADED stands if recall stays low WITH clean telemetry (stack then owes an explanation) - in which
case per-chunk forensics run on any missing probe golds.

Ingest was run via the CLI (init + `kgf ingest` on `config-h212-v2.yml`, neo4j pinned to the fresh
scratch instance `bolt://172.19.0.4:7687` per DEF-4/DEF-5), logged to `logs/h212-clean-recall.log`,
event log `logs/h212-v2-events.jsonl`. This notebook measures the resulting graph.

## Imports

In [1]:
# stdlib
import json
import datetime
import re
import collections
from pathlib import Path
# third party
from neo4j import GraphDatabase
from rich import print as rprint
from rich.table import Table
from rich.console import Console
# project (CWD = project root)
import sys
sys.path.insert(0, "src")
sys.path.insert(0, ".")
from notebooks.h158_measure import precision_proxy, recall_at_k, _present, _norm
console = Console()

2026-07-08 06:59:54.397 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


## Configuration

In [2]:
FRESH_URI = "bolt://172.19.0.4:7687"
FRESH_AUTH = ("neo4j", "kgfoundry")
H212_LOG = "logs/h212-v2-events.jsonl"
INGEST_LOG = "logs/h212-clean-recall.log"
BENCH = "reports/identity-benchmark-h101-20260707-094448.json"
PROBES = "tests/probes/cpap-probe-set.yml"
H212_RECALL_OUT = "reports/h212-v2-recall.json"
TOP_K = 16
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# H158 reference arms (the confound baseline)
H158 = {
    "v1_warnings": 76, "v2_warnings": 119,
    "v1_recall": 0.875, "v2_recall": 0.8333,
    "replay_false_merges": 11, "false_merge_tol": 0.20,
    "precision_bar": 0.50,
}
# Registered H212 bar
BAR_WARN = 76        # extraction.warning total must be <= this (v1 arm)
BAR_RECALL = 0.875   # recall@16 must be >= this (non-regression)

rprint("[bold cyan]R15-H212[/bold cyan]  stamp %s  bar: warnings<=%d AND recall>=%.3f"
       % (STAMP, BAR_WARN, BAR_RECALL))

R15-H212  stamp 20260708T045954Z  bar: warnings<=76 AND recall>=0.875

## Extraction telemetry - the registered failed-chunk warning count

`extraction.warning` events in the event log are the registered failed-chunk warnings. H158 v2 carried
119 (113 dangling-relationship + 6 `chunk extraction failed` = Bedrock quota casualties under wave-2
contention); v1 carried 76 (all dangling). Under quiet Bedrock the quota-casualty class should vanish
and the total should land at or below 76.

In [3]:
warn_reasons = collections.Counter()
warn_total = 0
warn_chunks = collections.Counter()
for line in Path(H212_LOG).read_text().splitlines():
    rec = json.loads(line)
    if rec.get("event") == "extraction.warning":
        warn_total += 1
        warn_reasons[rec.get("reason", "").split(":")[0]] += 1
        warn_chunks[rec.get("chunk", "?")] += 1

ev_hist = collections.Counter()
for line in Path(H212_LOG).read_text().splitlines():
    ev_hist[json.loads(line).get("event", "?")] += 1

rprint("[bold]extraction.warning total[/bold]: [yellow]%d[/yellow]  (H158 v1 %d / v2 %d)"
       % (warn_total, H158["v1_warnings"], H158["v2_warnings"]))
rprint("  by reason class: %s" % dict(warn_reasons))
chunk_fail = warn_reasons.get("chunk extraction failed", 0)
rprint("  quota-casualty class (chunk extraction failed): [yellow]%d[/yellow]  "
       "(H158 v2 had 6; clean Bedrock should be 0)" % chunk_fail)
rprint("  docs=%d/%d  skipped=%d  merges=%d  defers=%d  vetoes=%d"
       % (ev_hist.get("document.completed", 0), ev_hist.get("document.started", 0),
          ev_hist.get("document.skipped", 0), ev_hist.get("resolution.merge", 0),
          ev_hist.get("resolution.defer", 0), ev_hist.get("resolution.veto", 0)))

extraction.warning total: 82  (H158 v1 76 / v2 119)

by reason class: {'self-referencing relationship on AirFit F10': 1, 'self-referencing relationship on Quattro 
Air': 1, 'dangling relationship': 50, 'chunk extraction failed': 30}

quota-casualty class (chunk extraction failed): 30  (H158 v2 had 6; clean Bedrock should be 0)

docs=26/28  skipped=2  merges=527  defers=965  vetoes=91

## Graph census (fresh instance)

In [4]:
drv = GraphDatabase.driver(FRESH_URI, auth=FRESH_AUTH)
with drv.session() as s:
    ents = s.run("MATCH (e:Entity) RETURN count(e) AS c").single()["c"]
    rels = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    chunks = s.run("MATCH (c:Chunk) RETURN count(c) AS c").single()["c"]
rprint("entities %d  relationships %d  chunks %d  (H158 v2: 2201 ents / 6808 rels)"
       % (ents, rels, chunks))

entities 1585  relationships 5798  chunks 132  (H158 v2: 2201 ents / 6808 rels)

## SAME_AS precision proxy - resolver merges vs the 298 adjudicated labels
Recomputed from the H212 event log (transitive union of `resolution.merge` events).

In [5]:
pp = precision_proxy(H212_LOG, BENCH)
rprint("[bold]precision proxy[/bold]  merges %d  defers %d  vetoes %d  | tp %d fp %d fn %d"
       % (pp["merges_total"], pp["defers_total"], pp["nli_vetoes"],
          pp["proxy_tp"], pp["proxy_fp"], pp["proxy_fn"]))
rprint("  SAME_AS precision [yellow]%.4f[/yellow] (bar >= %.2f)  false-merges [yellow]%d[/yellow] (replay 11 +-20%%)"
       % (pp["same_as_precision"], H158["precision_bar"], pp["false_merges"]))

precision proxy  merges 527  defers 965  vetoes 91  | tp 15 fp 14 fn 37

SAME_AS precision 0.5172 (bar >= 0.50)  false-merges 14 (replay 11 +-20%)

## Pure-seed evidence recall@16 (24 gold probes)
Same harness as H158; neo4j uri pointed at the fresh instance, embeddings/graphrag from base `config.yml`
(Titan, unchanged).

In [6]:
rr = recall_at_k(FRESH_URI, TOP_K, "v2-h212", PROBES)
Path(H212_RECALL_OUT).write_text(json.dumps(rr, indent=2))
rprint("[bold]recall@%d[/bold]  mean [yellow]%.4f[/yellow] (bar >= %.3f; H158 v1 %.3f / v2 %.4f)  fully-covered %d/%d"
       % (TOP_K, rr["mean_recall"], BAR_RECALL, H158["v1_recall"], H158["v2_recall"],
          rr["fully_covered"], rr["n_probes"]))
failing = {k: v for k, v in rr["per_probe"].items() if v < 1.0}
rprint("  failing/partial probes: %s" % failing)

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


...

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


2026-07-08 07:00:02.110 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 1/1
2026-07-08 07:00:02.111 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - Generated embeddings for 1/1 entities via bedrock (0 cache hits)


recall@16  mean 0.7500 (bar >= 0.875; H158 v1 0.875 / v2 0.8333)  fully-covered 16/24

failing/partial probes: {'P08': 0.0, 'P09': 0.0, 'P14': 0.5, 'P15': 0.5, 'P16': 0.5, 'P17': 0.5, 'P21': 0.0, 
'P24': 0.0}

## Bar arithmetic and verdict

In [7]:
warn_ok = warn_total <= BAR_WARN
recall_ok = rr["mean_recall"] >= BAR_RECALL - 1e-9
clean_telemetry = warn_ok
fm_lo = H158["replay_false_merges"] * (1 - H158["false_merge_tol"])
fm_hi = H158["replay_false_merges"] * (1 + H158["false_merge_tol"])
fm_ok = fm_lo <= pp["false_merges"] <= fm_hi
prec_ok = pp["same_as_precision"] >= H158["precision_bar"]

if warn_ok and recall_ok:
    verdict = "CONFIRMED"
elif clean_telemetry and not recall_ok:
    verdict = "DEGRADED-stands"
else:
    verdict = "INCONCLUSIVE"

tbl = Table(title="R15-H212 - clean-conditions re-run vs H158 arms")
for h in ["metric", "H158 v1", "H158 v2", "H212", "bar", "pass"]:
    tbl.add_column(h)
tbl.add_row("extraction.warning total", str(H158["v1_warnings"]), str(H158["v2_warnings"]),
            str(warn_total), "<= %d" % BAR_WARN, "PASS" if warn_ok else "FAIL")
tbl.add_row("recall@16 (24 probes)", "%.4f" % H158["v1_recall"], "%.4f" % H158["v2_recall"],
            "%.4f" % rr["mean_recall"], ">= %.3f" % BAR_RECALL, "PASS" if recall_ok else "FAIL")
tbl.add_row("SAME_AS precision proxy", "-", "0.5217", "%.4f" % pp["same_as_precision"],
            ">= %.2f" % H158["precision_bar"], "PASS" if prec_ok else "FAIL")
tbl.add_row("false merges", "-", "11", str(pp["false_merges"]),
            "%.0f-%.0f" % (fm_lo, fm_hi), "PASS" if fm_ok else "FAIL")
console.print(tbl)
rprint("[bold]R15-H212 VERDICT: %s[/bold]" % verdict)

              R15-H212 - clean-conditions re-run vs H158 arms              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━┓
┃ metric                   ┃ H158 v1 ┃ H158 v2 ┃ H212   ┃ bar      ┃ pass ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━┩
│ extraction.warning total │ 76      │ 119     │ 82     │ <= 76    │ FAIL │
│ recall@16 (24 probes)    │ 0.8750  │ 0.8333  │ 0.7500 │ >= 0.875 │ FAIL │
│ SAME_AS precision proxy  │ -       │ 0.5217  │ 0.5172 │ >= 0.50  │ PASS │
│ false merges             │ -       │ 11      │ 14     │ 9-13     │ FAIL │
└──────────────────────────┴─────────┴─────────┴────────┴──────────┴──────┘

R15-H212 VERDICT: INCONCLUSIVE

## Per-chunk forensics on missing probe golds

Registered branch: if recall stays low with clean telemetry, adjudicate every missing gold across three
levels - is the gold string in the source chunks (readable document text), in the graph (extraction +
load captured it as an entity/property), and in the probe's top-16 seed render (retrieval reachability).
A gold present in chunks but absent from the graph is extraction loss; present in the graph but not in
seeds is a retrieval/identity miss. Runs only when at least one probe is below 1.0.

In [8]:
import yaml
probes = [p for p in yaml.safe_load(Path(PROBES).read_text()) if p.get("gold_evidence")]
probe_by_id = {p["id"]: p for p in probes}

def gold_in_chunks(session, gold):
    for r in session.run("MATCH (c:Chunk) RETURN c.text AS t"):
        if r["t"] and _present(gold, _norm(r["t"])):
            return True
    return False

def gold_in_graph(session, gold):
    for r in session.run("MATCH (e:Entity) RETURN e.name AS name, e.description AS d, properties(e) AS props"):
        spec = {k.removeprefix("prop_"): v for k, v in (r["props"] or {}).items() if k.startswith("prop_")}
        blob = _norm(" ".join([str(r["name"] or ""), str(r["d"] or ""), json.dumps(spec, default=str)]))
        if _present(gold, blob):
            return True
    return False

forensics = {}
if failing:
    from copy import deepcopy
    from knowledge_graph_foundry import load_settings
    from knowledge_graph_foundry.extraction import generate_embeddings
    from knowledge_graph_foundry.graph.graphrag import vector_query
    from knowledge_graph_foundry.models import Entity
    from knowledge_graph_foundry.pipeline import Foundry
    from notebooks.h158_measure import _render_nodes

    base = load_settings(Path("config.yml"))
    vec = base.graphrag.vector_index_name
    st = deepcopy(base)
    st.neo4j.uri, st.neo4j.user, st.neo4j.password = FRESH_URI, "neo4j", "kgfoundry"
    st.graphrag.top_k = TOP_K
    with Foundry(st) as f:
        for pid in failing:
            p = probe_by_id[pid]
            q = p["question"]
            pe = Entity.create(q[:80], types=["Query"], description=q)
            emb = generate_embeddings([pe], st.embeddings)[0].embedding
            seeds = [srow["id"] for srow in vector_query(f.driver, emb, vec, top_k=TOP_K)]
            with f.driver.session() as sess:
                ctx = _render_nodes(sess, seeds)
                per_gold = []
                for g in p["gold_evidence"]:
                    per_gold.append({
                        "gold": g,
                        "in_source_chunks": bool(gold_in_chunks(sess, g)),
                        "in_graph": bool(gold_in_graph(sess, g)),
                        "in_top16_seeds": bool(_present(g, ctx)),
                    })
            forensics[pid] = {"question": q, "sources": p.get("sources"),
                              "score": failing[pid], "golds": per_gold}
    for pid, fx in forensics.items():
        rprint("[bold]%s[/bold] (%.2f)  %s" % (pid, fx["score"], fx["question"][:70]))
        for g in fx["golds"]:
            ok = g["in_source_chunks"] and g["in_graph"] and g["in_top16_seeds"]
            flag = "" if ok else "  <-- MISS"
            rprint("    gold '%s'  chunk=%s graph=%s seeds=%s%s"
                   % (g["gold"], g["in_source_chunks"], g["in_graph"], g["in_top16_seeds"], flag))
else:
    rprint("[green]no failing probes - forensics not required[/green]")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


...

gold '0 to 45 min'  chunk=True graph=True seeds=True

P21 (0.00)  What is the purpose of the EZ-Start feature on the DreamStation CPAP P

gold 'gradually acclimate'  chunk=True graph=True seeds=False  <-- MISS

P24 (0.00)  On the ResMed AirSense 11, what does the EPR feature do during breathi

gold 'reduces the pressure during expiration'  chunk=True graph=True seeds=False  <-- MISS

## Write report

In [9]:
report = {
    "hypothesis": "R15-H212", "stamp": STAMP, "verdict": verdict,
    "fresh_instance": FRESH_URI, "config": "config-h212-v2.yml",
    "bar": {"warnings_le": BAR_WARN, "recall_ge": BAR_RECALL},
    "h158_reference": H158,
    "telemetry": {
        "extraction_warning_total": warn_total,
        "warning_by_reason": dict(warn_reasons),
        "chunk_extraction_failed": warn_reasons.get("chunk extraction failed", 0),
        "event_histogram": dict(ev_hist),
    },
    "census": {"entities": ents, "relationships": rels, "chunks": chunks},
    "precision_proxy": pp,
    "recall": rr,
    "clauses": {
        "warnings_le_76": bool(warn_ok),
        "recall_ge_0875": bool(recall_ok),
        "precision_ge_50pct": bool(prec_ok),
        "false_merges_within_tol": bool(fm_ok),
    },
    "forensics": forensics,
    "deviations": [],
}
out = Path("reports/clean-recall-h212-%s.json" % STAMP)
out.write_text(json.dumps(report, indent=2, default=str))
rprint("[bold]wrote[/bold] %s" % out)

wrote reports/clean-recall-h212-20260708T045954Z.json